# Making a yaml file
## optional
#### Yaml format 
#### note : paths should always have  / slash
#### train: path of images in the train folder
#### val: path of images in the val folder
#### nc : number of classes
#### names : names of the classes

In [14]:
#Renaming
import os

def batch_rename_images(directory_path, base_name):
    """
    Renames all images in a directory to a sequential base_name (e.g., base_name_1.jpg).
    """
    # Verify the directory exists
    if not os.path.exists(directory_path):
        print(f"Error: The directory '{directory_path}' does not exist.")
        return

    # List of common image extensions to ensure we only rename images
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
    
    # Get all files in the directory that match the image extensions
    files = [f for f in os.listdir(directory_path) if f.lower().endswith(valid_extensions)]
    
    # Sort files to ensure consistent order during renaming
    files.sort()
    
    if not files:
        print(f"No images found in '{directory_path}'.")
        return

    print(f"Found {len(files)} images. Starting renaming process...")

    for index, filename in enumerate(files, start=1):
        # Extract the original file extension (e.g., .jpg, .png)
        file_extension = os.path.splitext(filename)[1]
        
        # Create the new filename
        new_name = f"{base_name}_{index}{file_extension}"
        
        # Construct the full absolute or relative file paths
        old_filepath = os.path.join(directory_path, filename)
        new_filepath = os.path.join(directory_path, new_name)
        
        try:
            os.rename(old_filepath, new_filepath)
            print(f"Renamed: '{filename}' -> '{new_name}'")
        except FileExistsError:
            print(f"Skipped: '{new_name}' already exists. Please ensure an empty target state.")
        except Exception as e:
            print(f"Error renaming '{filename}': {e}")

    print("Renaming complete!")

# ==========================================
# CONFIGURATION VARIABLES
# ==========================================

# 1. Change this to the path where your images are currently saved. 
# You can use an absolute path (e.g., "C:/Users/Name/Images") or relative path.
DIRECTORY = r"C:\Users\ASUS\Desktop\yolo_train\pattern_dataset" 

# 2. Change this to your desired prefix. 
# The script will automatically append "_1", "_2", etc.
NEW_BASE_NAME = "blue_real"

# ==========================================

# Run the script (Uncomment the line below when ready to run)
batch_rename_images(DIRECTORY, NEW_BASE_NAME)

Found 115 images. Starting renaming process...
Renamed: 'fake_red_1.jpg' -> 'blue_real_1.jpg'
Renamed: 'fake_red_10.jpg' -> 'blue_real_2.jpg'
Renamed: 'fake_red_100.jpg' -> 'blue_real_3.jpg'
Renamed: 'fake_red_101.jpg' -> 'blue_real_4.jpg'
Renamed: 'fake_red_102.jpg' -> 'blue_real_5.jpg'
Renamed: 'fake_red_103.jpg' -> 'blue_real_6.jpg'
Renamed: 'fake_red_104.jpg' -> 'blue_real_7.jpg'
Renamed: 'fake_red_105.jpg' -> 'blue_real_8.jpg'
Renamed: 'fake_red_106.jpg' -> 'blue_real_9.jpg'
Renamed: 'fake_red_107.jpg' -> 'blue_real_10.jpg'
Renamed: 'fake_red_108.jpg' -> 'blue_real_11.jpg'
Renamed: 'fake_red_109.jpg' -> 'blue_real_12.jpg'
Renamed: 'fake_red_11.jpg' -> 'blue_real_13.jpg'
Renamed: 'fake_red_110.jpg' -> 'blue_real_14.jpg'
Renamed: 'fake_red_111.jpg' -> 'blue_real_15.jpg'
Renamed: 'fake_red_112.jpg' -> 'blue_real_16.jpg'
Renamed: 'fake_red_113.jpg' -> 'blue_real_17.jpg'
Renamed: 'fake_red_114.jpg' -> 'blue_real_18.jpg'
Renamed: 'fake_red_115.jpg' -> 'blue_real_19.jpg'
Renamed: 'fake_r

In [ ]:
import os
import yaml

def generate_yolo_yaml():
    # --- CONFIGURATION ---
    # 1. Where is your classes.txt? (Assumes it's in the same folder as this script)
    classes_file = r"C:\Users\ASUS\Desktop\yolo_train\new_Dataset\classes.txt"
    
    # 2. Output file name
    output_file = "data"
    
    # 3. Get absolute path of the current directory to avoid path errors
    current_dir = os.getcwd().replace("\\", "/")
    
    # --- LOGIC ---
     
    # Check if classes.txt exists
    if not os.path.exists(classes_file):
        # Fallback: Check if it's inside train/labels often found in Roboflow exports
        if os.path.exists("train/labels/classes.txt"):
            classes_file = "train/labels/classes.txt"
        else:
            print(f"Error: Could not find '{classes_file}'. Please place it next to this script.")
            return

    # Read class names
    with open(classes_file, 'r') as f:
        # distinct lines, stripped of whitespace
        class_names = [line.strip() for line in f.readlines() if line.strip()]

    print(f"Found {len(class_names)} classes: {class_names}")

    # Create the dictionary structure for YOLO
    data = {
        'path': current_dir,       # Dataset root dir
        'train': 'train/images',   # Train images (relative to 'path')
        'val': 'val/images',       # Val images (relative to 'path')
        
        # If you have a test folder, uncomment the next line:
        # 'test': 'test/images',
        
        'nc': len(class_names),    # Number of classes
        'names': class_names       # List of class names
    }

    # Write to data.yaml
    with open(output_file, 'w') as f:
        # Default flow_style=None makes it readable block format
        yaml.dump(data, f, sort_keys=False, default_flow_style=None)
        
    print(f"✅ Successfully created '{output_file}' at {current_dir}/{output_file}")
    print("Double check the 'train' and 'val' paths in the file if your folder structure is non-standard.")

if __name__ == "__main__":
    generate_yolo_yaml()

# Split  the dataset
### Splits the dataset into train and test sets

In [1]:
import os
import shutil
import random
from pathlib import Path

# ================= CONFIGURATION =================
# 1. Path to your IMAGES
IMG_DIR = r"C:\Users\ASUS\Desktop\yolo_train\pose_dataset\images"

# 2. Path to your LABELS (If same as images, paste the same path)
LBL_DIR = r"C:\Users\ASUS\Desktop\yolo_train\pose_dataset\labels"

# 3. Output folder
OUTPUT_DIR = "yolo_pose_dataset" 
# =================================================

def main():
    img_path_root = Path(IMG_DIR)
    lbl_path_root = Path(LBL_DIR)
    output_path = Path(os.getcwd()) / OUTPUT_DIR

    # 1. Collect Images
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
    image_files = []
    for ext in extensions:
        image_files.extend(list(img_path_root.rglob(ext)))
        # Also check uppercase extensions
        image_files.extend(list(img_path_root.rglob(ext.upper())))
    
    # Unique and sorted
    image_files = sorted(list(set(image_files)))

    if not image_files:
        print("❌ No images found. Check IMG_DIR path.")
        return

    print(f"✅ Found {len(image_files)} images.")

    # 2. Pair Images with Labels
    pairs = []
    missing_labels = 0

    print("   Pairing images with labels...")

    for img in image_files:
        stem = img.stem  # filename without extension (e.g. "cat" from "cat.jpg")
        
        # Look for the label file in the LBL_DIR
        # We try standard .txt first
        potential_label = lbl_path_root / f"{stem}.txt"
        
        # If not found, try the "double extension" style (e.g. cat.jpg.txt)
        if not potential_label.exists():
            potential_label = lbl_path_root / f"{img.name}.txt"

        if potential_label.exists():
            pairs.append((img, potential_label))
        else:
            missing_labels += 1
            # Uncomment below to see exactly which ones are failing
            # print(f"   ⚠️ Missing label for: {img.name}")

    if len(pairs) == 0:
        print("❌ CRITICAL: No matching labels found!")
        print(f"   Checked folder: {LBL_DIR}")
        print("   Make sure your text files have the EXACT same name as your images.")
        return

    print(f"✅ Successfully paired {len(pairs)} images with labels.")
    if missing_labels > 0:
        print(f"⚠️ Warning: {missing_labels} images were skipped because they had no labels.")

    # 3. Shuffle and Split
    random.seed(42)
    random.shuffle(pairs)

    split_idx = int(len(pairs) * 0.8)
    train_pairs = pairs[:split_idx]
    val_pairs = pairs[split_idx:]

    # 4. Copy Files
    dirs = {
        'train_img': output_path / 'train' / 'images',
        'train_lbl': output_path / 'train' / 'labels',
        'val_img': output_path / 'val' / 'images',
        'val_lbl': output_path / 'val' / 'labels'
    }

    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)

    def copy_data(pair_list, split_name):
        print(f"   Copying {len(pair_list)} pairs to {split_name}...")
        for img_src, lbl_src in pair_list:
            shutil.copy2(img_src, dirs[f'{split_name}_img'] / img_src.name)
            shutil.copy2(lbl_src, dirs[f'{split_name}_lbl'] / lbl_src.name)

    copy_data(train_pairs, 'train')
    copy_data(val_pairs, 'val')

    print("\n🎉 Done! Update your data.yaml to point to:", output_path)

if __name__ == "__main__":
    main()

✅ Found 27 images.
   Pairing images with labels...
✅ Successfully paired 27 images with labels.
   Copying 21 pairs to train...
   Copying 6 pairs to val...

🎉 Done! Update your data.yaml to point to: c:\Users\ASUS\Desktop\yolo_train\yolo_pose_dataset


## Training yolov8n model (For detecting boxes)

In [20]:
from ultralytics import YOLO
import torch

def main():
    # 1. Check if GPU is available
    if torch.cuda.is_available():
        print(f"GPU Name: {torch.cuda.get_device_name(0)}")
        print("Cuda is available. Training on RTX 4060.")
    else:
        print("Cuda not found. Training might be slow on CPU.")

  


   
model = YOLO('yolov8n-pose.pt')  # Load the Nano model

results = model.train(
    data='data_pose.yaml',
    epochs=50,
    imgsz=640,    # Keep resolution standard (640x640) for speed
    batch=16,
    device=0,     # Train on your laptop GPU
    
    # Augmentation: Essential because you aren't training on all 30 patterns yet,
    # just the "Box" shape. These help it generalize.
    degrees=10.0,
    scale=0.5,
    mosaic=1.0,
)
if __name__ == '__main__':
    # This guard is required for multiprocessing on Windows
    main()
    

New https://pypi.org/project/ultralytics/8.4.8 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.10.19 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_pose.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train14, nbs=64, nm

RuntimeError: Dataset 'data_pose.yaml' error  Dataset 'data_pose.yaml' images not found, missing path 'C:\Users\ASUS\Desktop\yolo_train\yolo_pose_dataset\C\Users\ASUS\Desktop\yolo_train\yolo_pose_dataset\val\images'
Note dataset download directory is 'C:\RAW\yolov8\datasets'. You can update this in 'C:\Users\ASUS\AppData\Roaming\Ultralytics\settings.json'

In [19]:
import os

base = "C:/Users/ASUS/Desktop/yolo_train/yolo_pose_dataset"
paths_to_check = [
    os.path.join(base, "train/images"),
    os.path.join(base, "train/labels"),
    os.path.join(base, "val/images"),
    os.path.join(base, "val/labels")
]

for p in paths_to_check:
    if os.path.exists(p):
        print(f"✅ Found: {p}")
        print(f"   Files inside: {len(os.listdir(p))}")
    else:
        print(f"❌ MISSING: {p}")

✅ Found: C:/Users/ASUS/Desktop/yolo_train/yolo_pose_dataset\train/images
   Files inside: 21
✅ Found: C:/Users/ASUS/Desktop/yolo_train/yolo_pose_dataset\train/labels
   Files inside: 21
✅ Found: C:/Users/ASUS/Desktop/yolo_train/yolo_pose_dataset\val/images
   Files inside: 6
✅ Found: C:/Users/ASUS/Desktop/yolo_train/yolo_pose_dataset\val/labels
   Files inside: 6


In [1]:
! pip install -U ultralytics

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
   ----------------- ---------------------- 0.5/1.2 MB 1.3 MB/s eta 0:00:01
   -------------------------- ------------- 0.8/1.2 MB 1.3 MB/s eta 0:00:01
   ----------------------------------- ---- 1.0/1.2 MB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 1.3 MB/s  0:00:00
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.3.237
    Uninstalling ultralytics-8.3.237:
      Successfully uninstalled ultralytics-8.3.237


### The trained model will be loacted at file path : runs\detect\train\weights

### Collect Dataset using the yolov8n model for the resnet model  

In [11]:
import cv2
from ultralytics import YOLO
import os


def main():
    # --- CONFIGURATION ---
    # Path to your "Finder" model (trained to find just the box)
    model_path = r"C:\Users\ASUS\Desktop\yolo_train\finder.pt"

    # Input video file (Walk around the Real/Fake scrolls)
    video_path = r"C:\Users\ASUS\Desktop\yolo_train\blue_real.mp4"

    # Output folder for the crops
    # CHANGE THIS NAME for each pattern you record!
    # e.g., 'dataset_reader/train/real_oracle_1'
    output_folder = r"C:\Users\ASUS\Desktop\yolo_train\pattern_dataset"

    # Confidence to accept a crop
    conf_threshold = 0.6
    # ---------------------

    # 1. Setup
    model = YOLO(model_path)
    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    count = 0

    print(f"Starting to crop from {video_path}...")
    print(f"Saving to: {output_folder}")

    while True:
        success, frame = cap.read()
        if not success:
            break

        # 2. Run Finder YOLO
        results = model(frame, conf=conf_threshold, verbose=False)

        for r in results:
            boxes = r.boxes
            for box in boxes:
                # Get coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)

                # Check if box is reasonable size (ignore tiny noise)
                w, h = x2 - x1, y2 - y1
                if w < 50 or h < 50:
                    continue

                # 3. Crop and Save
                crop = frame[y1:y2, x1:x2]

                # Verify crop isn't empty
                if crop.size == 0:
                    continue

                filename = f"{output_folder}/crop_{count}.jpg"
                cv2.imwrite(filename, crop)
                count += 1

                # Draw on frame just for visualization
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        cv2.imshow("Auto-Cropper", frame)

        # Press 'q' to quit early
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f"✅ Finished! Saved {count} cropped images.")


if __name__ == "__main__":
    main()


Starting to crop from C:\Users\ASUS\Desktop\yolo_train\blue_real.mp4...
Saving to: C:\Users\ASUS\Desktop\yolo_train\pattern_dataset
✅ Finished! Saved 367 cropped images.


## Training (Resnet model) classifies real and fake patterns


In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import os

def main():
    # --- CONFIGURATION ---
    DATA_DIR = r'C:\Users\ASUS\Desktop\yolo_train\reader_dataset'  # Folder containing your class folders
    BATCH_SIZE = 32
    EPOCHS = 50
    LEARNING_RATE = 0.001
    # ---------------------

    # 1. Check Hardware
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Training on: {device}")

    # 2. Data Transformations (Augmentation)
    # We add noise/rotation to make the robot robust to vibration/angles
    train_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(15),       # Handle tilted scrolls
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2), # Handle lighting
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # 3. Load Data
    if not os.path.exists(DATA_DIR):
        print(f"❌ Error: Folder '{DATA_DIR}' not found!")
        return

    full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transforms)
    
    # Calculate classes
    class_names = full_dataset.classes
    num_classes = len(class_names)
    print(f"📂 Found {num_classes} Classes: {class_names}")

    # Auto-split 80% Train, 20% Validation
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
    
    # Update validation transform (hacky but works for simple splits)
    val_dataset.dataset.transform = val_transforms

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    print(f"📊 Data Split: {len(train_dataset)} Training, {len(val_dataset)} Validation")

    # 4. Setup Model (MobileNetV3-Small)
    print("🧠 Loading MobileNetV3-Small...")
    model = models.mobilenet_v3_small(weights='DEFAULT')

    # Modify the last layer to match YOUR 30 classes (default is 1000)
    # The classifier structure in MobileNetV3-Small ends with Linear(1024, 1000)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, num_classes)

    model = model.to(device)

    # 5. Training Loop
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print("\n🏁 Starting Training...")
    
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        # Validation phase
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        train_acc = 100 * correct / total
        val_acc = 100 * val_correct / val_total
        
        print(f"Epoch [{epoch+1}/{EPOCHS}] "
              f"Loss: {running_loss/len(train_loader):.4f} | "
              f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    # 6. Save the Model
    save_path = "reader_best.pth"
    torch.save(model.state_dict(), save_path)
    print(f"\n✅ Training Complete! Model saved to '{save_path}'")
    
    # Save class names too (useful for the robot)
    with open("classes.txt", "w") as f:
        for c in class_names:
            f.write(c + "\n")
    print("📝 Class names saved to 'classes.txt'")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

def evaluate_and_plot(model, val_loader, class_names, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 1. Classification Report (Precision, Recall, F1)
    print("\nDetailed Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # 2. Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('confusion_matrix.png')
    plt.show()
    print("📈 Confusion matrix saved as 'confusion_matrix.png'")
if __name__ == "__main__":
    main()

🚀 Training on: cuda
📂 Found 3 Classes: ['blue_real', 'fake_red', 'real_red']
📊 Data Split: 436 Training, 109 Validation
🧠 Loading MobileNetV3-Small...

🏁 Starting Training...
Epoch [1/50] Loss: 0.1138 | Train Acc: 94.04% | Val Acc: 95.41%
Epoch [2/50] Loss: 0.0000 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch [3/50] Loss: 0.0000 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch [4/50] Loss: 0.0000 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch [5/50] Loss: 0.0000 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch [6/50] Loss: 0.0000 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch [7/50] Loss: 0.0000 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch [8/50] Loss: 0.0000 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch [9/50] Loss: 0.0000 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch [10/50] Loss: 0.0000 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch [11/50] Loss: 0.0000 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch [12/50] Loss: 0.0000 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch [13/50] Loss: 0.0000 | Tra

## converting reader to onnx form

In [6]:
import torch
import torch.nn as nn
from torchvision import models

def export_onnx():
    # --- CONFIGURATION ---
    WEIGHTS_PATH = "C:/Users/ASUS/Desktop/yolo_train/reader.pth"
    OUTPUT_PATH = "reader.onnx"
    NUM_CLASSES = 17  # MUST match your training folder count!
    # ---------------------

    print(f"Loading {WEIGHTS_PATH}...")
    
    # 1. Re-create the architecture exactly as you trained it
    model = models.mobilenet_v3_small(weights=None)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, NUM_CLASSES)
    
    # 2. Load the trained weights
    try:
        model.load_state_dict(torch.load(WEIGHTS_PATH, map_location="cpu"))
    except Exception as e:
        print(f"❌ Error loading weights: {e}")
        return

    model.eval() 

    # 3. Create a dummy input (1 image, 3 channels, 224x224)
    dummy_input = torch.randn(1, 3, 224, 224)

    # 4. Export
    print(f"Exporting to {OUTPUT_PATH}...")
    torch.onnx.export(
        model, 
        dummy_input, 
        OUTPUT_PATH,
        export_params=True,
        opset_version=12,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
    )
    print("✅ Success! Copy 'reader.onnx' to your robot.")

if __name__ == "__main__":
    export_onnx()

Loading C:/Users/ASUS/Desktop/yolo_train/reader.pth...
Exporting to reader.onnx...


C:\Users\ASUS\AppData\Local\Temp\ipykernel_23012\3506724748.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=

✅ Success! Copy 'reader.onnx' to your robot.


## Converting finder to onnnx format

In [1]:
# Replace 'best.pt' with the path to your YOLO finder model
! yolo export model="C:/Users/ASUS/Desktop/yolo_train/finder.pt" task=detect format=onnx

Ultralytics 8.4.0  Python-3.10.19 torch-2.5.1+cu121 CPU (13th Gen Intel Core(TM) i7-13620H)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'C:\Users\ASUS\Desktop\yolo_train\finder.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (6.0 MB)

ONNX: starting export with onnx 1.20.0 opset 19...
ONNX: slimming with onnxslim 0.1.82...
ONNX: export success  0.8s, saved as 'C:\Users\ASUS\Desktop\yolo_train\finder.onnx' (11.7 MB)

Export complete (1.1s)
Results saved to C:\Users\ASUS\Desktop\yolo_train
Predict:         yolo predict task=detect model=C:\Users\ASUS\Desktop\yolo_train\finder.onnx imgsz=640  
Validate:        yolo val task=detect model=C:\Users\ASUS\Desktop\yolo_train\finder.onnx imgsz=640 data=C:/Users/ASUS/Desktop/yolo_train/data.yaml  
Visualize:       https://netron.app
 Learn more at https://docs.ultralytics.com/modes/export


## Black and white reader training


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import os

def main():
    # --- CONFIGURATION ---
    DATA_DIR = 'dataset_reader/train' 
    BATCH_SIZE = 32
    EPOCHS = 60
    # ---------------------

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Training Grayscale Model on: {device}")

    # 1. Grayscale Transforms
    train_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=1),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])

    # 2. Load Data
    if not os.path.exists(DATA_DIR):
        print(f"❌ Error: Folder '{DATA_DIR}' not found!")
        return

    full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transforms)
    num_classes = len(full_dataset.classes)
    
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    print(f"📂 Found {num_classes} Classes. Train: {train_size}, Val: {val_size}")

    # 3. Setup Model
    model = models.mobilenet_v3_small(weights='DEFAULT')

    # Modify First Layer for 1 Channel
    original_layer = model.features[0][0]
    model.features[0][0] = nn.Conv2d(
        in_channels=1,
        out_channels=original_layer.out_channels,
        kernel_size=original_layer.kernel_size,
        stride=original_layer.stride,
        padding=original_layer.padding,
        bias=False
    )
    
    # Modify Classifier
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, num_classes)
    model = model.to(device)

    # 4. Train & Validate
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print("🏁 Starting Training...")
    
    for epoch in range(EPOCHS):
        # --- TRAINING PHASE ---
        model.train()
        train_loss, train_correct = 0.0, 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            train_correct += (predicted == labels).sum().item()

        train_loss = train_loss / len(train_loader.dataset)
        train_acc = (train_correct / len(train_loader.dataset)) * 100

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss, val_correct = 0.0, 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == labels).sum().item()

        val_loss = val_loss / len(val_loader.dataset)
        val_acc = (val_correct / len(val_loader.dataset)) * 100

        print(f"Epoch [{epoch+1}/{EPOCHS}] "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")

    # 5. Save
    torch.save(model.state_dict(), "reader_gray.pth")
    print("\n✅ Saved 'reader_gray.pth'")
    
    with open("classes.txt", "w") as f:
        for c in full_dataset.classes:
            f.write(c + "\n")

if __name__ == "__main__":
    main()

🚀 Training Grayscale Model on: cuda
📂 Found 17 Classes. Train: 1240, Val: 310
🏁 Starting Training...
Epoch [1/60] Train Loss: 0.7835 Acc: 78.71% | Val Loss: 2.7984 Acc: 18.39%
Epoch [2/60] Train Loss: 0.0755 Acc: 97.82% | Val Loss: 2.5795 Acc: 35.81%
Epoch [3/60] Train Loss: 0.0315 Acc: 98.79% | Val Loss: 3.2173 Acc: 21.29%
Epoch [4/60] Train Loss: 0.0081 Acc: 99.76% | Val Loss: 2.5159 Acc: 30.97%
Epoch [5/60] Train Loss: 0.0147 Acc: 99.84% | Val Loss: 2.4549 Acc: 37.10%
Epoch [6/60] Train Loss: 0.0248 Acc: 99.27% | Val Loss: 1.5316 Acc: 62.58%
Epoch [7/60] Train Loss: 0.0122 Acc: 99.68% | Val Loss: 0.8179 Acc: 72.58%
Epoch [8/60] Train Loss: 0.0057 Acc: 99.92% | Val Loss: 0.6132 Acc: 84.52%
Epoch [9/60] Train Loss: 0.0115 Acc: 99.52% | Val Loss: 0.7520 Acc: 77.10%
Epoch [10/60] Train Loss: 0.0135 Acc: 99.44% | Val Loss: 1.7535 Acc: 61.61%
Epoch [11/60] Train Loss: 0.0243 Acc: 99.35% | Val Loss: 1.5090 Acc: 68.71%
Epoch [12/60] Train Loss: 0.0425 Acc: 98.95% | Val Loss: 1.3673 Acc: 70.

## Black and White reader conversion to onnnx


In [7]:
import torch
import torch.nn as nn
from torchvision import models

def export_onnx():
    # --- CONFIGURATION ---
    WEIGHTS_PATH = "reader_gray.pth"
    OUTPUT_PATH = "reader_gray.onnx"
    NUM_CLASSES = 17 # Must match your classes.txt count
    # ---------------------

    print(f"Loading {WEIGHTS_PATH}...")
    
    # 1. Initialize Standard Model
    model = models.mobilenet_v3_small(weights=None)
    
    # 2. CRITICAL: Change First Layer to 1 Channel (Grayscale)
    # This MUST happen BEFORE loading weights!
    original_layer = model.features[0][0]
    model.features[0][0] = nn.Conv2d(
        in_channels=1, # <--- 1 Channel input
        out_channels=original_layer.out_channels,
        kernel_size=original_layer.kernel_size,
        stride=original_layer.stride,
        padding=original_layer.padding,
        bias=False
    )
    
    # 3. Change Last Layer (Classifier)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, NUM_CLASSES)
    
    # 4. Load Weights (Now shapes will match!)
    try:
        # weights_only=True is safer, but if it fails, remove it
        state_dict = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=True)
        model.load_state_dict(state_dict)
        print("✅ Weights loaded successfully.")
    except Exception as e:
        print(f"❌ Error loading weights: {e}")
        # Fallback for older PyTorch versions
        state_dict = torch.load(WEIGHTS_PATH, map_location="cpu")
        model.load_state_dict(state_dict)

    model.eval() 

    # 5. Create Dummy Input (1 Channel)
    dummy_input = torch.randn(1, 1, 224, 224) 

    # 6. Export
    print(f"Exporting to {OUTPUT_PATH}...")
    torch.onnx.export(
        model, 
        dummy_input, 
        OUTPUT_PATH,
        export_params=True,
        opset_version=12,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
    )
    print(f"✅ Success! Created {OUTPUT_PATH}")

if __name__ == "__main__":
    export_onnx()

Loading reader_gray.pth...
✅ Weights loaded successfully.
Exporting to reader_gray.onnx...


OnnxExporterError: Module onnx is not installed!

In [8]:
! pip install onnx

  Using cached ml_dtypes-0.5.4-cp310-cp310-win_amd64.whl.metadata (9.2 kB)
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.4 MB 4.2 MB/s eta 0:00:04
   --- ------------------------------------ 1.3/16.4 MB 4.2 MB/s eta 0:00:04
   ------ --------------------------------- 2.6/16.4 MB 4.4 MB/s eta 0:00:04
   -------- ------------------------------- 3.7/16.4 MB 4.6 MB/s eta 0:00:03
   ----------- ---------------------------- 4.7/16.4 MB 4.8 MB/s eta 0:00:03
   --------------- ------------------------ 6.3/16.4 MB 5.1 MB/s eta 0:00:02
   ----------------- ---------------------- 7.3/16.4 MB 5.3 MB/s eta 0:00:02
   --------------------- ------------------ 8.7/16.4 MB 5.4 MB/s eta 0:00:02
   ------------------------ --------------- 10.0/16.4 MB 5.4 MB/s eta 0:00:02
   --------------------------- ------------ 11.3/16.4 MB 5.5 MB/s eta 0:00:01
   ------------------------------ --------- 12.6/16.4 MB 5.6 MB/s eta 0:00:01
   --

In [21]:
from ultralytics import YOLO
import torch

def main():
    # 1. Double-check CUDA is available for your RTX 4060
    device = 0 if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {torch.cuda.get_device_name(0) if device == 0 else 'CPU'}")

    # 2. Load the Pose model
    model = YOLO('yolov8n-pose.pt') 

    # 3. Start training
    results = model.train(
        data='C:/Users/ASUS/Desktop/yolo_train/data_pose.yaml',
        epochs=50,
        imgsz=640,
        batch=16,
        device=device,
        workers=0,  # Prevents Windows multiprocessing crashes
        exist_ok=True # Overwrites existing 'train' folders in 'runs'
    )

if __name__ == '__main__':
    main()

Using device: NVIDIA GeForce RTX 4060 Laptop GPU
New https://pypi.org/project/ultralytics/8.4.8 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.10.19 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/Users/ASUS/Desktop/yolo_train/data_pose.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8

RuntimeError: Dataset 'C://Users/ASUS/Desktop/yolo_train/data_pose.yaml' error  Dataset 'C://Users/ASUS/Desktop/yolo_train/data_pose.yaml' images not found, missing path 'C:\Users\ASUS\Desktop\yolo_train\yolo_pose_dataset\C\Users\ASUS\Desktop\yolo_train\yolo_pose_dataset\val\images'
Note dataset download directory is 'C:\RAW\yolov8\datasets'. You can update this in 'C:\Users\ASUS\AppData\Roaming\Ultralytics\settings.json'